In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.special import expit, comb
from sklearn.metrics import (
    roc_auc_score, f1_score,
    brier_score_loss, log_loss,
    average_precision_score, accuracy_score
)
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

DATA_DIR = Path("../data/email-Enron")
EDGE_CSV = DATA_DIR / "enron_order2_edges.csv"   # columns i,j

edges = pd.read_csv(EDGE_CSV)
E = np.sort(edges[["i","j"]].values.astype(int), axis=1)
E = np.unique(E, axis=0)  # dedup

print("Loaded unique 2-edges:", E.shape[0])

# Reindex nodes to 0..n-1
nodes = np.unique(E.reshape(-1))
node2new = {int(u): idx for idx, u in enumerate(nodes)}
E = np.vectorize(node2new.get)(E)

n = len(nodes)
m = E.shape[0]
print("Num nodes:", n)


In [ ]:
def node_degrees_from_pairs(edges: np.ndarray, n_nodes: int):
    """Compute the degree of each node from an (m, 2) array of edges."""
    deg = np.zeros(n_nodes, dtype=int)
    np.add.at(deg, edges.reshape(-1), 1)
    return deg

def make_edge_set(E: np.ndarray):
    """Build a Python set of edge tuples for O(1) membership tests."""
    return set(map(tuple, E.tolist()))

def predict_edge_probs(beta, edges):
    """Compute link probabilities expit(sum_{v in e} beta_v) for a batch of edges."""
    s = np.sum(beta[edges], axis=1)
    return expit(s)

def sample_negative_edges(n, forbidden_set, m, rng, r=2):
    """Uniformly sample m vertex r-tuples not present in forbidden_set, for use
    as negative (non-edge) examples."""
    neg = []
    seen = set()
    tries = 0
    max_tries = 50 * m + 10_000
    while len(neg) < m and tries < max_tries:
        tries += 1
        edge = rng.choice(n, size=r, replace=False)
        edge.sort()
        t = tuple(map(int, edge))
        if t in forbidden_set or t in seen:
            continue
        seen.add(t)
        neg.append(t)
    return np.array(neg, dtype=np.int32)

def ece_score(y_true, y_prob, n_bins=15):
    """Compute the expected calibration error (ECE) of predicted probabilities
    y_prob against binary outcomes y_true, using n_bins equal-width bins."""
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for b in range(n_bins):
        mask = (y_prob >= bins[b]) & (y_prob < bins[b+1])
        if not np.any(mask): continue
        ece += mask.mean() * abs(y_true[mask].mean() - y_prob[mask].mean())
    return float(ece)

def evaluate_metrics(y_true, p_hat, thresholds=(0.5, 0.33, 0.25)):
    """Compute ROC-AUC, average precision, Brier score, log-loss, ECE, and
    F1/accuracy at each threshold in `thresholds`."""
    out = {
        "roc_auc": roc_auc_score(y_true, p_hat),
        "ap": average_precision_score(y_true, p_hat),
        "brier": brier_score_loss(y_true, p_hat),
        "logloss": log_loss(y_true, np.clip(p_hat, 1e-15, 1-1e-15)),
        "ece": ece_score(y_true, p_hat)
    }
    for th in thresholds:
        y_pred = (p_hat >= th).astype(int)
        out[f"f1@{th:.2f}"] = f1_score(y_true, y_pred, zero_division=0)
        out[f"acc@{th:.2f}"] = accuracy_score(y_true, y_pred)
    return out

def discrete_laplace_noise(eps_per_coordinate, size, rng=None):
    """Sample iid discrete Laplace noise with parameter a = exp(-eps_per_coordinate).
    For an r-uniform hypergraph degree sequence, set eps_per_coordinate = eps / r."""
    rng = np.random.default_rng() if rng is None else rng
    a = np.exp(-eps_per_coordinate)
    mag = rng.geometric(p=1 - a, size=size) - 1
    sign = rng.choice([-1, 1], size=size)
    return mag * sign


In [ ]:
class HypergraphBeta2:
    """Log-partition function, gradients, and degree-based estimators for the
    2-uniform (ordinary graph) beta-model, computed densely since r=2 admits
    an O(n^2) rather than chunked O(n^r) computation."""

    def __init__(self, n):
        self.n = n
        self.r = 2
        self.C = comb(n, 2)

    def A_and_grad(self, beta):
        """Compute A(beta) = sum_{i<j} log(1+exp(beta_i+beta_j)) and its gradient
        (the vector of expected degrees)."""
        n = self.n
        A = 0.0
        grad = np.zeros(n, dtype=float)
        for i in range(n - 1):
            s = beta[i] + beta[i+1:]
            A += np.logaddexp(0.0, s).sum()
            p = expit(s)
            grad[i] += p.sum()
            grad[i+1:] += p
        return A, grad

    def ridge_box_fit_from_degrees(self, d_obs, M, lam, beta_init=None,
                                    maxiter=2000, verbose=False):
        """Fit beta via projected gradient descent on the ridge-regularized negative
        log-likelihood given observed (possibly noisy) degrees d_obs, then clip to
        [-M, M]. The true M is unknown for real data, so this uses a fixed,
        empirically-stable step size rather than the theoretical worst-case rate.
        """
        n = self.n
        if beta_init is None:
            beta_init = np.zeros(n, dtype=float)

        beta = beta_init.copy()
        eta = 0.005

        if verbose:
            print(f"Starting GD: eta={eta}, iterations={maxiter}")

        for t in range(maxiter):
            _, gA = self.A_and_grad(beta)
            grad_obj = (gA - d_obs + lam * beta) / self.C

            beta = beta - eta * grad_obj

            if verbose and t % 500 == 0:
                print(f"Iter {t}: grad_norm={np.linalg.norm(grad_obj)}")

        beta_truncated = np.clip(beta, -M, M)

        return beta_truncated, None

    def box_mle_fit_from_degrees(self, d_obs, M, beta_init=None,
                                maxiter=10000, verbose=False):
        """Box-constrained MLE: special case of ridge_box_fit_from_degrees with lam=0."""
        return self.ridge_box_fit_from_degrees(
            d_obs=d_obs, M=M, lam=0.0, beta_init=beta_init,
            maxiter=maxiter, verbose=verbose
        )

    def grad_ell(self, beta, d_true, chunk_size_pairs=250_000):
        """Gradient of the (unregularized) negative log-likelihood at beta,
        given the observed degree sequence d_true."""
        _, gA = self.A_and_grad(beta)
        return (gA - d_true) / self.C


In [ ]:
def central_dp_gd(model, d_true, n, r, M, eps, delta, T_cap=10000, seed=None, verbose=False):
    """Differentially private gradient descent estimator for beta under central
    (eps, delta)-edge differential privacy: adds Gaussian noise to the gradient
    at each step, with iteration count and noise scale calibrated so the final
    iterate is (eps, delta)-DP (step size fixed at 0.005, as above)."""
    rng = np.random.default_rng(seed)
    beta = np.zeros(n)
    eta = 0.005
    T = min(T_cap, int(np.ceil(32.0*(r-1)*np.exp(4.0*r*M)*((r-1)*np.log(n)+2.0*np.log(M)))))
    sigma = np.sqrt(4.0 * r * T * (n**(-2.0*r)) * (eps**-2.0) * np.log(1.0/delta))

    for t in range(T):
        if verbose and ((t + 1) % 500 == 0 or (t + 1) == T):
            print(f"Iter {t+1}/{T} eps={eps}", flush=True)
        g = model.grad_ell(beta, d_true)
        beta -= eta * (g + rng.normal(0, sigma, size=n))
    return np.clip(beta, -M, M), {"T": T, "sigma": sigma}


def local_dp_fit_from_degrees(model, d_true, M, lam, eps_loc, seed=0,
                              maxiter=10000, verbose=False):
    """Local DP: privatize the degree vector d_true with discrete Laplace noise,
    then fit the ridge/box estimator using the privatized degrees."""
    rng = np.random.default_rng(seed)
    z = discrete_laplace_noise(eps_per_coordinate=eps_loc, size=len(d_true), rng=rng)
    d_priv = d_true.astype(float) + z.astype(float)
    d_priv = np.maximum(d_priv, 0.0)

    beta_loc, _ = model.ridge_box_fit_from_degrees(
        d_obs=d_priv, M=M, lam=lam, beta_init=None,
        maxiter=maxiter, verbose=verbose
    )
    return beta_loc, {"eps_loc": float(eps_loc)}


In [ ]:
r = 2
model = HypergraphBeta2(n)
M = np.sqrt(np.log(n)) * 2.0
delta = n**(-2.0)
lam_ridge = 0.0002 * (n**((r-1)/2))

# Hold out 10% of observed edges as test positives; the rest form the training
# graph. Test negatives are an equal number of non-edges.
rng = np.random.default_rng(123)

perm = rng.permutation(m)
m_test_pos = int(np.ceil(0.10 * m))
test_pos_idx = perm[:m_test_pos]
train_pos_idx = perm[m_test_pos:]

E_train = E[train_pos_idx]   # observed edges for fitting
E_test_pos = E[test_pos_idx] # held-out positives

# Forbid sampling any observed positive (including held-out) as a negative
obs_set = make_edge_set(E)
forbidden_test = set(obs_set)
E_test_neg = sample_negative_edges(n, forbidden_test, E_test_pos.shape[0], rng)

print("Train edges:", E_train.shape[0])
print("Test positives:", E_test_pos.shape[0])
print("Test negatives:", E_test_neg.shape[0])


d_train = node_degrees_from_pairs(E_train, n).astype(float)
print("Mean train degree:", d_train.mean(), "Max:", d_train.max())

results = []


In [ ]:
M = np.sqrt(np.log(n)) * 2.0
maxiter_mle = 10000
chunk_size_pairs = 250_000

delta = n ** (-2.0)
lam_ridge = 0.0002 * (n ** ((r - 1) / 2))   # for r=2, this is 0.0002

print("delta =", delta)
print("lam_ridge =", lam_ridge)

# Build the unified test set once
E_test = np.vstack([E_test_pos, E_test_neg])
y_test = np.concatenate([
    np.ones(E_test_pos.shape[0], dtype=int),
    np.zeros(E_test_neg.shape[0], dtype=int)
])

# ---- Method 1: Non-private MLE (box-constrained) ----
beta_mle, _ = model.box_mle_fit_from_degrees(
    d_obs=d_train, M=M, beta_init=None,
    maxiter=maxiter_mle, verbose=True
)

# ---- Method 2: Non-private ridge MLE (box-constrained) ----
beta_mle_lam, _ = model.ridge_box_fit_from_degrees(
    d_obs=d_train, M=M, lam=lam_ridge, beta_init=None,
    maxiter=maxiter_mle, verbose=True
)

# Evaluate non-private baselines once
p_mle = predict_edge_probs(beta_mle, E_test)
p_mle_lam = predict_edge_probs(beta_mle_lam, E_test)

rows = []
rows.append({"method": "MLE", "eps": np.nan, "delta": np.nan, "lam": 0.0, "M": M, **evaluate_metrics(y_test, p_mle)})
rows.append({"method": "MLE_lam", "eps": np.nan, "delta": np.nan, "lam": lam_ridge, "M": M, **evaluate_metrics(y_test, p_mle_lam)})

# ---- Methods 3 & 4: DP methods over an eps grid ----
eps_grid = [0.001, 0.01, 0.1, 1.0]

for eps in eps_grid:
    # Central DP-GD
    beta_cen, info_cen = central_dp_gd(
        model=model, d_true=d_train, n=n, r=r, M=M, eps=eps, delta=delta,
        seed=999, T_cap=10000, verbose=True
    )
    p_cen = predict_edge_probs(beta_cen, E_test)
    rows.append({
        "method": "CEN_DP",
        "eps": eps,
        "delta": delta,
        "lam": 0.0,
        "M": M,
        "T": info_cen.get("T", np.nan),
        "sigma": info_cen.get("sigma", np.nan),
        **evaluate_metrics(y_test, p_cen)
    })

    # Local DP (privatize degrees then fit ridge/box)
    beta_loc, info_loc = local_dp_fit_from_degrees(
        model=model, d_true=d_train, M=M, lam=lam_ridge, eps_loc=eps,
        seed=2026, maxiter=maxiter_mle, verbose=True
    )
    p_loc = predict_edge_probs(beta_loc, E_test)
    rows.append({
        "method": "LOC_DP",
        "eps": eps,
        "delta": np.nan,     # local DP does not use delta in this degree-noise construction
        "lam": lam_ridge,
        "M": M,
        **evaluate_metrics(y_test, p_loc)
    })

res = pd.DataFrame(rows)
res


In [ ]:
res.to_csv("link_prediction_results_order2.csv", index=False)
